# CreditSense Public-Best Pipeline

This notebook is the clean public reproduction of the **public-best Kaggle lane** from the project: `meta_overnight`. It keeps the strong tree ensemble, a lighter MPS neural branch, and the fixed historical blend/meta settings so the workflow stays readable.

## Run mode

- `RUN_SMOKE_TEST = True` keeps execution fast and is the mode used for the committed notebook output.
- Switch it to `False` to run the full public-best pipeline on the complete training data.

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from creditsense_public import KAGGLE_COMPETITION_URL, PipelineConfig, run_public_best_pipeline

RUN_SMOKE_TEST = True
KAGGLE_COMPETITION_URL


'https://www.kaggle.com/t/3e62a127eb85418aa851a5ee258e7c04'

In [2]:
results = run_public_best_pipeline(PipelineConfig(smoke_test=RUN_SMOKE_TEST))
score_table = results["score_table"].copy()
score_table

/opt/anaconda3/lib/python3.12/site-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=2.13366e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/opt/anaconda3/lib/python3.12/site-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=2.23402e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/opt/anaconda3/lib/python3.12/site-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=2.62269e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/opt/anaconda3/lib/python3.12/site-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.01931e-10): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/opt/anaconda3/lib/python3.12/site-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=9.00839e-11): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)


,model,accuracy,r2,combined
0,meta_overnight,0.834571,0.849399,0.841985
1,blend_overnight,0.826857,0.844459,0.835658
2,lgb,0.785000,0.839957,0.812478
3,mps,0.809000,0.812382,0.810691
4,xgb,0.773429,0.837447,0.805438
5,cat,0.758857,0.842835,0.800846
6,et,0.702143,0.763567,0.732855


## Historical reference

The original research repo recorded a **public-best local validation score** of `0.849489` for `meta_overnight`. The smoke run below is only a correctness check, so its score is expected to be lower.

In [3]:
submission_preview = results["submission"].head(10).copy()
submission_preview

,Id,RiskTier,InterestRate
0,0,2,6.450000
1,1,0,6.010000
2,2,2,6.180000
3,3,3,6.230000
4,4,4,23.129999
5,5,3,6.570000
6,6,2,6.130000
7,7,2,6.050000
8,8,2,6.200000
9,9,2,6.350000


## Branch notes

- `xgb`, `lgb`, `cat`, and `et` are fixed-parameter tree families derived from the historical tuning logs.
- `mps` is a compact multitask neural branch used to keep the public notebook readable while still capturing the original diversity idea.
- `blend_overnight` uses the historical learned weights.
- `meta_overnight` fits the historical logistic-regression and ridge meta models on the current validation predictions.